# BiLSTM + FastText - Turkce Metin Siniflandirma Kiyaslamasi

**Katman:** Derin (transformer disi)

FastText gomme uzerine BiLSTM - PeerJ'deki MConv-BiLSTM-GAM ile karsilastirilabilirlik.

Gorev: Turkce deprem ihbar metni -> 5 ust kategori (+ opsiyonel 1-5 aciliyet).  
Veri: `data/` (bkz `data/README.md`).  
Butun defterler `common.py` uzerinden AYNI bolme ve AYNI metrikleri kullanir.  
Metrik: Dogruluk, makro-F1, sinif bazli P/R, konfuzyon matrisi, **yanlis negatif orani**.

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/04_turkce_metin
    %pip install -q torch
print('cwd:', os.getcwd())

## 1. Veri

In [ ]:
import sys, warnings; warnings.filterwarnings('ignore')
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd
import common as C

train_df, val_df, test_df = C.load_text_dataset()
labels, l2i, i2l = C.label_maps(train_df)
print('siniflar:', labels)
print('train/val/test:', len(train_df), len(val_df), len(test_df))
y_test = test_df['label'].map(l2i).values

## 2. Model

In [ ]:
MODEL_NAME = "bilstm_fasttext"
PARAMS = dict(embed_dim=100, hidden=128, epochs=10, lr=1e-3, batch_size=32, max_len=60)
import torch, torch.nn as nn
from collections import Counter

def tokenize(s): return s.lower().split()
cnt = Counter(w for t in train_df['text'] for w in tokenize(t))
vocab = {'<pad>': 0, '<unk>': 1, **{w: i+2 for i, (w, _) in enumerate(cnt.most_common(20000))}}
def encode(s):
    ids = [vocab.get(w, 1) for w in tokenize(s)][:PARAMS['max_len']]
    return ids + [0]*(PARAMS['max_len']-len(ids))

class BiLSTM(nn.Module):
    def __init__(self, V, E, H, C):
        super().__init__()
        self.emb = nn.Embedding(V, E, padding_idx=0)
        self.lstm = nn.LSTM(E, H, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(2*H, C)
    def forward(self, x):
        o, _ = self.lstm(self.emb(x))
        return self.fc(o.mean(1))

## 3. Egitim

In [ ]:
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
Xtr = torch.tensor([encode(t) for t in train_df['text']]); ytr = torch.tensor(train_df['label'].map(l2i).values)
Xte = torch.tensor([encode(t) for t in test_df['text']])
net = BiLSTM(len(vocab), PARAMS['embed_dim'], PARAMS['hidden'], len(labels)).to(dev)
opt = torch.optim.Adam(net.parameters(), lr=PARAMS['lr']); lossf = nn.CrossEntropyLoss()
for ep in range(PARAMS['epochs']):
    net.train(); perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), PARAMS['batch_size']):
        idx = perm[i:i+PARAMS['batch_size']]
        opt.zero_grad()
        out = net(Xtr[idx].to(dev)); loss = lossf(out, ytr[idx].to(dev))
        loss.backward(); opt.step()
    print(f'epoch {ep+1} loss {loss.item():.4f}')

## 4. Sonuc

In [ ]:
net.eval()
with torch.no_grad(): y_pred = net(Xte.to(dev)).argmax(1).cpu().numpy()

In [ ]:
m = C.classification_metrics(y_test, y_pred, labels)
print('accuracy :', round(m['accuracy'], 4))
print('macro-F1 :', round(m['macro_f1'], 4))
print('macro yanlis-negatif orani:', round(m['macro_fnr'], 4))
print()
print(m['report'])
lb = C.save_result(MODEL_NAME, m, extra={'params': PARAMS})
lb

In [ ]:
import matplotlib.pyplot as plt
cm = np.array(m['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
for (i, j), v in np.ndenumerate(cm): ax.text(j, i, int(v), ha='center', va='center')
ax.set_xlabel('tahmin'); ax.set_ylabel('gercek'); ax.set_title(f'{MODEL_NAME} konfuzyon')
plt.tight_layout(); plt.savefig(f'results/{MODEL_NAME}_cm.png', dpi=140); plt.show()